# #73 2016~2024 예산 데이터 계보 및 입력 완전성 점검

## tl;dr

- `필터링전_전체원본`, wide, long은 153개 연도·시도 조합이 모두 있다. wide 57,979행과 long 115,958행(2행/사업)의 키·사업명·예산은 일치한다.
- `필터링전` 세부사업 후보는 57,987행으로 wide보다 8행 많다. 모두 2022년의 제목·합계·구분 오탐지 후보로, wide에 추가된 행은 없다.
- 칼럼정렬 Excel은 2016~2020·2022~2024년 8개만 가독 가능하다. 2021년 파일은 165 bytes이며 Excel 포맷이 아니므로 원본 재확보가 필요하다.
- Excel→CSV 키 계보는 2020·2022~2024년에 완전히 일치한다. 2016~2018년 CSV는 Excel의 부분집합이고, 공통 키에서 예산 불일치 행이 각 6·3·3건 있다. 2019년은 양방향 키 차이와 사업명 불일치 98건이 있어 변환 규칙·원본 버전 확인이 필요하다.
- 따라서 `필터링전` CSV는 wide/long의 직접 선행 자료로는 검증됐지만, 2016~2019년에서 원본 Excel을 대체하는 독립 신뢰 기준으로 바로 쓰면 안 된다.


## Context & Methods

### Key Assumptions

- 분석 단위는 `연도·지역·원본행`의 세부사업 1행이다.
- `필터링전_전체원본.csv`는 연도별 칼럼정렬 Excel의 `정리본_자동` 시트에서 지역별로 나눈 중간 산출물이라는 가설을 키·사업명·예산 대조로 검증한다.
- wide는 `사업행구분=세부사업`인 원본 행의 정제 산출물이며, long은 wide 예산을 당해·전년도 행으로 풀어쓴 산출물이라는 가설을 검증한다.
- 이 노트북은 계보·입력 완전성 점검이며, 예산 오연결 후보 판정은 후속 노트북/스크립트에서 수행한다.


## Data

### 1. 경로와 공통 함수

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()
)
INTERIM_ROOT = PROJECT_ROOT / "data/interim"
RAW_EXCEL_ROOT = PROJECT_ROOT / "data/raw/칼럼정렬"
YEARS = tuple(range(2016, 2025))
REGIONS = (
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주",
)
EXPECTED_KEYS = {(year, region) for year in YEARS for region in REGIONS}
FILE_KINDS = ("필터링전_전체원본", "세부사업_정제", "세부사업_정제_long")


def nfc(value):
    return unicodedata.normalize("NFC", str(value))


def normalize_source_row(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def normalize_text(series):
    return series.fillna("").astype(str).map(nfc).str.replace(r"\s+", " ", regex=True).str.strip()


def numeric(series):
    text = series.astype("string").str.replace(",", "", regex=False).str.strip()
    text = text.replace({"": pd.NA, "-": pd.NA, "(신규)": pd.NA, "(추가)": pd.NA})
    return pd.to_numeric(text, errors="coerce")


def numeric_equal(left, right):
    left_num = numeric(left)
    right_num = numeric(right)
    return (left_num.isna() & right_num.isna()) | np.isclose(
        left_num.fillna(0), right_num.fillna(0), rtol=0, atol=1e-9
    )


def csv_path(year, region, kind):
    return INTERIM_ROOT / region / f"{year}_{region}_{kind}.csv"


def read_csv(path):
    return pd.read_csv(path, encoding="utf-8-sig", low_memory=False, dtype={"원본행": "string"})


def year_from_name(path):
    matched = re.search(r"20(?:1[6-9]|2[0-4])", nfc(path.name))
    return int(matched.group()) if matched else None

### 2. 153개 조합의 파일 존재와 Excel 가독성

In [2]:
inventory_rows = []
for year, region in sorted(EXPECTED_KEYS):
    row = {"연도": year, "지역": region}
    for kind in FILE_KINDS:
        path = csv_path(year, region, kind)
        row[f"{kind}_존재"] = path.exists()
        row[f"{kind}_크기"] = path.stat().st_size if path.exists() else 0
    inventory_rows.append(row)

inventory = pd.DataFrame(inventory_rows)
inventory_summary = pd.DataFrame(
    {
        "자료": FILE_KINDS,
        "존재_조합": [int(inventory[f"{kind}_존재"].sum()) for kind in FILE_KINDS],
        "기대_조합": 153,
    }
)
display(inventory_summary)
display(inventory.loc[~inventory[[f"{kind}_존재" for kind in FILE_KINDS]].all(axis=1)])

excel_rows = []
for path in sorted(RAW_EXCEL_ROOT.glob("*.xlsx")):
    if nfc(path.name).startswith("~$"):
        continue
    year = year_from_name(path)
    if year not in YEARS:
        continue
    readable = False
    sheets = []
    error = ""
    try:
        excel = pd.ExcelFile(path, engine="openpyxl")
        sheets = excel.sheet_names
        readable = "정리본_자동" in sheets
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
    excel_rows.append(
        {
            "연도": year,
            "파일": path.name,
            "크기_bytes": path.stat().st_size,
            "가독_가능": readable,
            "시트": sheets,
            "오류": error,
        }
    )
excel_inventory = pd.DataFrame(excel_rows).sort_values("연도")
display(excel_inventory)

,자료,존재_조합,기대_조합
0,필터링전_전체원본,153,153
1,세부사업_정제,153,153
2,세부사업_정제_long,153,153


,연도,지역,필터링전_전체원본_존재,필터링전_전체원본_크기,세부사업_정제_존재,세부사업_정제_크기,세부사업_정제_long_존재,세부사업_정제_long_크기


,연도,파일,크기_bytes,가독_가능,시트,오류
0,2016,세부사업표추출_2016년도 지방자치단체 저추...,2452522,True,"[Table 1, 정리본_자동, 검증_자동]",
1,2017,세부사업표추출_2017년도 지방자치단체 저추...,2174054,True,"[Table 1, 정리본_자동, 검증_자동]",
2,2018,세부사업표추출_2018년도 지방자치단체 저추...,3213939,True,"[Table 1, 정리본_자동, 검증_자동]",
3,2019,세부사업표추출_2019년도 지방자치단체 저추...,1868552,True,"[Table 1, 정리본_자동, 검증_자동]",
4,2020,세부사업표추출_2020년도 지방자치단체 저추...,1949098,True,"[Table 1, 정리본_자동, 검증_자동]",
8,2021,세부사업표추출_2021년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획)_...,165,False,[],BadZipFile: File is not a zip file
5,2022,세부사업표추출_2022년도 지방자치단체 저추...,3156842,True,"[Table 1, 정리본_자동, 검증_자동]",
6,2023,세부사업표추출_2023년도 지방자치단체 저추...,2909951,True,"[Table 1, 정리본_자동, 검증_자동]",
7,2024,세부사업표추출_2024년도 지방자치단체 저추...,2585988,True,"[Table 1, 정리본_자동, 검증_자동]",


## Results

### 3. CSV 스키마·행수·키 범위 프로파일

In [3]:
profile_rows = []
for year, region in sorted(EXPECTED_KEYS):
    for kind in FILE_KINDS:
        path = csv_path(year, region, kind)
        if not path.exists():
            continue
        frame = read_csv(path)
        key_missing = None
        key_duplicates = None
        wrong_year = None
        wrong_region = None
        if "원본행" in frame:
            source_row = normalize_source_row(frame["원본행"])
            key_missing = int(source_row.isna().sum() + source_row.eq("").sum())
            key_duplicates = int(source_row.duplicated(keep=False).sum())
        if "연도" in frame:
            wrong_year = int(pd.to_numeric(frame["연도"], errors="coerce").ne(year).sum())
        if "지역" in frame:
            wrong_region = int(normalize_text(frame["지역"]).ne(region).sum())
        profile_rows.append(
            {
                "연도": year,
                "지역": region,
                "자료": kind,
                "행수": len(frame),
                "컬럼수": len(frame.columns),
                "원본행_결측": key_missing,
                "원본행_중복행": key_duplicates,
                "연도_불일치": wrong_year,
                "지역_불일치": wrong_region,
                "컬럼": tuple(frame.columns),
            }
        )

profiles = pd.DataFrame(profile_rows)
display(
    profiles.groupby("자료").agg(
        조합=("자료", "size"), 총행=("행수", "sum"), 최소행=("행수", "min"), 최대행=("행수", "max")
    )
)
profile_issues = profiles.loc[
    (profiles["원본행_결측"].fillna(0) > 0)
    | ((profiles["자료"] != "세부사업_정제_long") & (profiles["연도_불일치"].fillna(0) > 0))
    | (profiles["지역_불일치"].fillna(0) > 0)
]
display(profile_issues)

,조합,총행,최소행,최대행
자료,,,,
세부사업_정제,153,57979,73,1315
세부사업_정제_long,153,115958,146,2630
필터링전_전체원본,153,61089,85,1348


,연도,지역,자료,행수,컬럼수,원본행_결측,원본행_중복행,연도_불일치,지역_불일치,컬럼
24,2016,서울,필터링전_전체원본,119,18,1,0,NaN,0,"(세부사업명, 사업분류재정구분, 2016년 예산, 2015년 예산, 증감액, 비율,..."
75,2017,서울,필터링전_전체원본,146,18,1,0,NaN,0,"(세부사업명, 사업분류재정구분, 2017년 예산, 2016년 예산, 증감액, 비율,..."
126,2018,서울,필터링전_전체원본,208,18,1,0,NaN,0,"(세부사업명, 사업분류재정구분, 2018년 예산, 2017년 예산, 증감액, 비율,..."
177,2019,서울,필터링전_전체원본,179,18,1,0,NaN,0,"(세부사업명, 사업분류재정구분, 2019년 예산, 2018년 예산, 증감액, 비율,..."
228,2020,서울,필터링전_전체원본,186,19,1,0,NaN,0,"(세부사업명, 사업분류재정구분, 2020년 예산, 2019년 예산, 증감액, 비율,..."


### 4. `필터링전` 세부사업 ↔ wide ↔ long 계보 대조

In [4]:
lineage_rows = []
for year, region in sorted(EXPECTED_KEYS):
    raw_path = csv_path(year, region, "필터링전_전체원본")
    wide_path = csv_path(year, region, "세부사업_정제")
    long_path = csv_path(year, region, "세부사업_정제_long")
    raw = read_csv(raw_path)
    leaf = raw.loc[raw["사업행구분"].eq("세부사업")].copy()
    leaf["원본행"] = normalize_source_row(leaf["원본행"])
    long = read_csv(long_path)
    long["원본행"] = normalize_source_row(long["원본행"])
    long_keys = set(long["원본행"])
    leaf_keys = set(leaf["원본행"])

    result = {
        "연도": year,
        "지역": region,
        "필터링전_세부사업행": len(leaf),
        "long_고유키": long["원본행"].nunique(),
        "필터링전만_키": len(leaf_keys - long_keys),
        "long만_키": len(long_keys - leaf_keys),
        "wide_존재": wide_path.exists(),
    }
    if wide_path.exists():
        wide = read_csv(wide_path)
        wide["원본행"] = normalize_source_row(wide["원본행"])
        wide_keys = set(wide["원본행"])
        merged = leaf.merge(
            wide, on="원본행", how="outer", suffixes=("_원본", "_wide"), indicator=True
        )
        both = merged["_merge"].eq("both")
        name_match = normalize_text(merged.loc[both, "세부사업명_원본"]).eq(
            normalize_text(merged.loc[both, "세부사업명_wide"])
        )
        current_col = f"{year}년 예산"
        previous_col = f"{year - 1}년 예산"
        current_match = (
            numeric_equal(merged.loc[both, current_col], merged.loc[both, "당해예산"])
            if current_col in merged
            else pd.Series(dtype=bool)
        )
        previous_match = (
            numeric_equal(merged.loc[both, previous_col], merged.loc[both, "전년도예산"])
            if previous_col in merged
            else pd.Series(dtype=bool)
        )
        result.update(
            {
                "wide_행": len(wide),
                "필터링전만_wide키": len(leaf_keys - wide_keys),
                "wide만_키": len(wide_keys - leaf_keys),
                "사업명_불일치": int((~name_match).sum()),
                "당해예산_불일치": int((~current_match).sum()) if len(current_match) else None,
                "전년도예산_불일치": int((~previous_match).sum()) if len(previous_match) else None,
            }
        )
    lineage_rows.append(result)

lineage = pd.DataFrame(lineage_rows)
display(lineage.sum(numeric_only=True).to_frame("합계").T)
issue_columns = [
    "필터링전만_키",
    "long만_키",
    "필터링전만_wide키",
    "wide만_키",
    "사업명_불일치",
    "당해예산_불일치",
    "전년도예산_불일치",
]
display(lineage.loc[lineage[issue_columns].fillna(0).gt(0).any(axis=1)].head(30))

,연도,필터링전_세부사업행,long_고유키,필터링전만_키,long만_키,wide_존재,wide_행,필터링전만_wide키,wide만_키,사업명_불일치,당해예산_불일치,전년도예산_불일치
합계,309060,57987,57979,8,0,153,57979,8,0,0,0,0


,연도,지역,필터링전_세부사업행,long_고유키,필터링전만_키,long만_키,wide_존재,wide_행,필터링전만_wide키,wide만_키,사업명_불일치,당해예산_불일치,전년도예산_불일치
103,2022,경기,1311,1310,1,0,True,1310,1,0,0,0,0
110,2022,서울,228,227,1,0,True,227,1,0,0,0,0
112,2022,울산,255,254,1,0,True,254,1,0,0,0,0
113,2022,인천,378,377,1,0,True,377,1,0,0,0,0
114,2022,전남,553,552,1,0,True,552,1,0,0,0,0
115,2022,전북,499,498,1,0,True,498,1,0,0,0,0
117,2022,충남,623,622,1,0,True,622,1,0,0,0,0
118,2022,충북,493,492,1,0,True,492,1,0,0,0,0


### 5. Excel `정리본_자동` ↔ `필터링전` 지역별 CSV 대조

In [5]:
excel_lineage_rows = []
for excel_row in excel_inventory.itertuples(index=False):
    if not excel_row.가독_가능:
        continue
    year = int(excel_row.연도)
    path = RAW_EXCEL_ROOT / excel_row.파일
    excel_frame = pd.read_excel(
        path, sheet_name="정리본_자동", engine="openpyxl", dtype={"원본행": "string"}
    )
    excel_frame["원본행"] = normalize_source_row(excel_frame["원본행"])
    excel_frame["지역"] = normalize_text(excel_frame["지역"])
    for region in REGIONS:
        csv_frame = read_csv(csv_path(year, region, "필터링전_전체원본"))
        csv_frame["원본행"] = normalize_source_row(csv_frame["원본행"])
        excel_region = excel_frame.loc[excel_frame["지역"].eq(region)].copy()
        excel_keys = set(excel_region["원본행"])
        csv_keys = set(csv_frame["원본행"])
        merged = excel_region.merge(
            csv_frame, on="원본행", how="inner", suffixes=("_excel", "_csv")
        )
        name_match = normalize_text(merged["세부사업명_excel"]).eq(
            normalize_text(merged["세부사업명_csv"])
        )
        current_col = f"{year}년 예산"
        previous_col = f"{year - 1}년 예산"
        current_match = numeric_equal(merged[f"{current_col}_excel"], merged[f"{current_col}_csv"])
        previous_match = numeric_equal(
            merged[f"{previous_col}_excel"], merged[f"{previous_col}_csv"]
        )
        excel_lineage_rows.append(
            {
                "연도": year,
                "지역": region,
                "Excel_행": len(excel_region),
                "CSV_행": len(csv_frame),
                "Excel만_키": len(excel_keys - csv_keys),
                "CSV만_키": len(csv_keys - excel_keys),
                "사업명_불일치": int((~name_match).sum()),
                "당해예산_불일치": int((~current_match).sum()),
                "전년도예산_불일치": int((~previous_match).sum()),
            }
        )

excel_lineage = pd.DataFrame(excel_lineage_rows)
display(
    excel_lineage.groupby("연도").agg(
        조합=("지역", "size"),
        Excel_행=("Excel_행", "sum"),
        CSV_행=("CSV_행", "sum"),
        Excel만_키=("Excel만_키", "sum"),
        CSV만_키=("CSV만_키", "sum"),
        사업명_불일치=("사업명_불일치", "sum"),
        당해예산_불일치=("당해예산_불일치", "sum"),
        전년도예산_불일치=("전년도예산_불일치", "sum"),
    )
)
excel_issue_columns = [
    "Excel만_키",
    "CSV만_키",
    "사업명_불일치",
    "당해예산_불일치",
    "전년도예산_불일치",
]
display(excel_lineage.loc[excel_lineage[excel_issue_columns].gt(0).any(axis=1)].head(30))

,조합,Excel_행,CSV_행,Excel만_키,CSV만_키,사업명_불일치,당해예산_불일치,전년도예산_불일치
연도,,,,,,,,
2016,17,6882,4802,2080,0,0,6,6
2017,17,11808,4873,6935,0,2,3,3
2018,17,8291,5998,2293,0,2,3,3
2019,17,8377,6774,2042,439,98,0,0
2020,17,7003,7003,0,0,0,0,0
2022,17,8472,8472,0,0,1,0,0
2023,17,7712,7712,0,0,0,0,0
2024,17,7340,7340,0,0,0,0,0


,연도,지역,Excel_행,CSV_행,Excel만_키,CSV만_키,사업명_불일치,당해예산_불일치,전년도예산_불일치
0,2016,서울,217,119,98,0,0,0,0
1,2016,부산,740,570,170,0,0,0,0
2,2016,대구,282,180,102,0,0,0,0
3,2016,인천,190,148,42,0,0,0,0
4,2016,광주,452,289,163,0,0,2,2
5,2016,대전,444,257,187,0,0,1,1
6,2016,울산,247,173,74,0,0,0,0
7,2016,세종,192,131,61,0,0,0,0
8,2016,경기,269,155,114,0,0,2,2
9,2016,강원,358,268,90,0,0,0,0


## Takeaways

1. 2020·2022~2024년은 Excel `정리본_자동`과 `필터링전` CSV의 키가 완전히 일치하므로 원본 예산 대조 파일럿을 바로 진행할 수 있다.
2. 2016~2018년은 CSV가 Excel 부분집합이 된 필터링 규칙과 예산 불일치 12행의 원인을 먼저 확인한다.
3. 2019년은 키·사업명 차이가 커서 현재 Excel과 CSV가 동일 버전에서 파생됐는지 확인한다.
4. 2021년은 유효한 칼럼정렬 Excel을 재확보하거나, 17개 시도 라벨 Excel이 원예산 근거를 충분히 보존했는지 별도 검증한다.
5. 이 선행 조치 후 57,979행 전수 오연결 감사로 확장한다. wide/long 자체의 예산 일치는 확인됐지만, 이는 원출처 예산의 정확성을 독립적으로 증명하지는 않는다.
